# Data Source 05 · 에너지·환경

| 출처 | 접근 | 검증 상태 |
|---|---|---|
| NOAA GML Mauna Loa CO₂ 월별 (`co2_mm_mlo.csv`) | 무키 | A |
| Open-Meteo Historical Weather (ERA5 재분석) | 무키 | A (일평균 변수명은 응답으로 확인) |
| Open-Meteo Air Quality | 무키 | A (한국 지점은 해상도 낮음, 예측 자료는 실측 아님) |
| Open Power System Data / UCI Household Power | 수동 | B / A |
| 기상청 KMA / AirKorea | 무료키·신청 | C |

NOAA 파일은 `#` 주석 행이 있고 결측이 **−99.99**로 표시된다. Open-Meteo는 **관측소 값이 아니라 재분석 격자 값**이다.

In [ ]:
import io, json, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt

def fetch_bytes(url, timeout=30):
    """무키 공개 URL 요청. 실패 시 예외 -> 호출부에서 합성 대체."""
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

def synthetic_series(n, freq, seed, trend=0.05, season=12, noise=1.0, start="2015-01-01"):
    """NOT real data: 방법 시연용 합성 시계열."""
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    y = 100 + trend * t + 3 * np.sin(2 * np.pi * t / season) + rng.normal(0, noise, n)
    return pd.Series(y, index=pd.date_range(start, periods=n, freq=freq), name="value")

def quality_report(s):
    s = s.sort_index()
    print("관측 수:", len(s), "| 기간:", s.index.min(), "~", s.index.max())
    print("결측:", int(s.isna().sum()), "| 중복 시점:", int(s.index.duplicated().sum()))
    f = pd.infer_freq(s.index) if len(s) > 2 else None
    print("추정 주기:", f)

In [ ]:
def parse_noaa_co2(raw):
    text = raw.decode("utf-8") if isinstance(raw, (bytes, bytearray)) else raw
    df = pd.read_csv(io.StringIO(text), comment="#")
    df.columns = [c.strip().lower() for c in df.columns]
    if not {"year", "month", "average"}.issubset(df.columns):
        raise ValueError("예상 열 없음: " + ", ".join(df.columns))
    s = df["average"].where(df["average"] > -90)          # -99.99 -> NaN
    idx = pd.to_datetime(dict(year=df["year"], month=df["month"], day=1))
    return pd.Series(s.values, index=idx, name="co2_ppm")

def parse_openmeteo_daily(obj):
    d = obj["daily"]
    tmax = pd.Series(d["temperature_2m_max"], index=pd.to_datetime(d["time"]))
    tmin = pd.Series(d["temperature_2m_min"], index=pd.to_datetime(d["time"]))
    return ((tmax + tmin) / 2).rename("t_mid")             # 최고·최저 평균: 정의된 일평균이 아님

In [ ]:
try:
    co2 = parse_noaa_co2(fetch_bytes("https://gml.noaa.gov/aftp/products/trends/co2/co2_mm_mlo.csv"))
    DATA_KIND = "REAL: NOAA GML Mauna Loa CO2"
except Exception as e:
    print("NOAA 수집 실패 -> 합성 대체:", type(e).__name__, e)
    co2 = synthetic_series(720, "MS", 2026051, trend=0.15, season=12, noise=0.3, start="1965-01-01")
    DATA_KIND = "SYNTHETIC (NOT real data)"
print("DATA_KIND (CO2):", DATA_KIND); quality_report(co2); co2.plot(title=DATA_KIND); plt.show()

# 선택: Open-Meteo 서울 일 기온 (무키, 비상업 무료, 호출 제한 있음)
try:
    u = ("https://archive-api.open-meteo.com/v1/archive?latitude=37.57&longitude=126.98"
         "&start_date=2020-01-01&end_date=2023-12-31&daily=temperature_2m_max,temperature_2m_min&timezone=Asia%2FSeoul")
    t = parse_openmeteo_daily(json.loads(fetch_bytes(u, timeout=60)))
    print("REAL (ERA5 재분석, 관측소 아님):", len(t), "일"); t.plot(title="Seoul t_mid (Open-Meteo/ERA5)"); plt.show()
except Exception as e:
    print("Open-Meteo 실패 (선택 항목):", type(e).__name__, e)

## 점검 체크리스트
- 재분석/모델 값과 관측소 실측을 구분해 보고한다
- 결측 코드(−99.99 등)와 시간대(UTC/현지)
- 예측(forecast) 자료를 과거 실측처럼 쓰지 않는다
- 국내 자료(KMA, AirKorea, KPX)는 키·이용약관 확인 후 추가